# Graph of Convex Sets — simple test

Minimal end-to-end test of the GCS-based free-space navigation using the apartment world.

## Navigate PR2 through free space using GCS

In [1]:
import math
import time
import pathlib
from dotenv import load_dotenv
from semantic_digital_twin.world_description.geometry import BoundingBox
from semantic_digital_twin.world_description.shape_collection import BoundingBoxCollection
from semantic_digital_twin.world_description.world_entity import SemanticEnvironmentAnnotation
from semantic_digital_twin.spatial_types import HomogeneousTransformationMatrix, Point3
from semantic_digital_twin.world_description.graph_of_convex_sets import GraphOfConvexSets
from generative_backend import load_pr2_apartment_world

load_dotenv(pathlib.Path().resolve().parent / ".env", override=True)

world, pr2, context = load_pr2_apartment_world()
print("World loaded")

Found these qp solvers: ['qpSWIFT', 'qpalm']


/home/malineni/workingdir/cognitive_robot_abstract_machine/probabilistic_model/src/probabilistic_model/probabilistic_model.py:239: SyntaxWarning: invalid escape sequence '\i'
  """
/home/malineni/workingdir/cognitive_robot_abstract_machine/probabilistic_model/src/probabilistic_model/probabilistic_circuit/rx/probabilistic_circuit.py:538: SyntaxWarning: invalid escape sequence '\_'
  """
Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
INFO:world.py::601 add_kinematic_structure_entity Trying to add kinematic_structure_entity with name pr2/base_footprint
INFO:world.py::601 add_kinematic_structure_entity Trying to add 

World loaded


In [2]:
import rclpy
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher

rclpy.init()
_ros_node = rclpy.create_node('semantic_digital_twin')
import threading
_ros_thread = threading.Thread(target=rclpy.spin, args=(_ros_node,), daemon=True)
_ros_thread.start()

In [3]:
world.get_body_by_name('island_back').global_pose.to_position().to_np()

array([2.90321, 2.69596, 0.468  , 1.     ])

In [4]:
# Use the apartment subtree of the loaded world as the obstacle annotation.
# This scopes GCS obstacle detection to apartment furniture only (excludes PR2).
#
# The apartment URDF has collision geometry at counter height (z ≈ 0.8–1.5).
# We set the search space at z=0.8–1.2 so GCS finds those real collision boxes.
# Path planning uses z=1.0 (inside search space); robot execution uses z=0 (floor).

apt_root = world.get_body_by_name("apartment_root")
apt_annotation = SemanticEnvironmentAnnotation(root=apt_root, _world=world)

nav_search_space = BoundingBoxCollection(
    [
        BoundingBox(
            min_x=-0.1, max_x=4.0,
            min_y=-0.1, max_y=4.0,
            min_z=0.9,  max_z=1.1,
            origin=HomogeneousTransformationMatrix(reference_frame=world.root),
        )
    ],
    world.root,
)

nav_gcs = GraphOfConvexSets.free_space_from_semantic_annotation(
    search_space=nav_search_space,
    semantic_obstacle_annotation=apt_annotation,
    tolerance=0.001,
)
print(f"Navigation GCS built — {len(nav_gcs.graph)} nodes")

INFO:graph_of_convex_sets.py::418 free_space_from_semantic_annotation Free space calculated in 336.46438 ms
INFO:graph_of_convex_sets.py::436 free_space_from_semantic_annotation Connectivity calculated in 24.405922 ms


Navigation GCS built — 593 nodes


In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2,  specs=[[{'type': 'surface'}, {'type': 'surface'}]], subplot_titles=["Occupied Space", "Free Space"])

occupied_traces = nav_gcs.plot_occupied_space()
fig.add_traces(occupied_traces, rows=[1 for _ in occupied_traces], cols=[1 for _ in occupied_traces])
free_traces = nav_gcs.plot_free_space()
fig.add_traces(free_traces, rows=[1 for _ in free_traces], cols=[2 for _ in free_traces])
fig.show()

In [6]:
_tf_publisher = TFPublisher(_world=world, node=_ros_node)
_viz_publisher = VizMarkerPublisher(_world=world, node=_ros_node)
print("ROS2 publishers started")

ROS2 publishers started


In [ ]:
# # Inspect world body positions to calibrate obstacle box placements.
# # Filters to bodies whose names contain kitchen/island/counter/cabinet keywords.
# keywords = ("island", "counter", "cabinet", "kitchen", "oven", "sink", "fridge")
#
# print(f"{'Body':<40} {'global position (x, y, z)'}")
# print("-" * 70)
# for body in world.bodies:
#     name = str(body.name)
#     if any(k in name.lower() for k in keywords):
#         try:
#             pos = body.global_pose.to_position().to_np()
#             print(f"{name:<40} ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")
#         except Exception:
#             print(f"{name:<40} (pose unavailable)")

In [9]:
# Robot starts between the two counter lines.
# Goal is to the right of the island — GCS routes around kitchen obstacles.
drive_conn = world.get_body_by_name("base_footprint").parent_connection
current_origin = drive_conn.origin
start_x = float(current_origin[0, 3])
start_y = float(current_origin[1, 3])

goal_x, goal_y = start_x+0.5 , start_y

print(f"Start: ({start_x:.2f}, {start_y:.2f})  →  Goal: ({goal_x}, {goal_y})")
print("Right counter/island blocks direct path — GCS routes around it")

# Plan at z=1.0 (counter height where collision boxes exist in the URDF)
start_pt = Point3(start_x, start_y, 1.0, reference_frame=world.root)
goal_pt  = Point3(goal_x,  goal_y,  1.0, reference_frame=world.root)

nav_path = nav_gcs.path_from_to(start_pt, goal_pt)

if nav_path is None:
    print("No path found — goal may be inside an obstacle or disconnected region")
else:
    print(f"\nPath found — {len(nav_path)} waypoints:")
    for i, p in enumerate(nav_path):
        print(f"  [{i}] ({float(p.x):.3f}, {float(p.y):.3f})")


Start: (1.00, 2.00)  →  Goal: (1.5, 2.0)
Right counter/island blocks direct path — GCS routes around it

Path found — 2 waypoints:
  [0] (1.000, 2.000)
  [1] (1.500, 2.000)


In [10]:
assert nav_path is not None, "Run the path planning cell first"

drive_conn = world.get_body_by_name("base_footprint").parent_connection

step_delay = 0.9  # seconds between waypoints

for i, waypoint in enumerate(nav_path):
    # Path was planned at z=1.0 (counter height); robot moves at floor level (z=0)
    wx, wy = float(waypoint.x), float(waypoint.y)

    if i < len(nav_path) - 1:
        nx, ny = float(nav_path[i + 1].x), float(nav_path[i + 1].y)
        yaw = math.atan2(ny - wy, nx - wx)
    else:
        yaw = 0.0

    drive_conn.origin = HomogeneousTransformationMatrix.from_xyz_rpy(wx, wy, 0.0, 0.0, 0.0, yaw)
    world.notify_state_change()

    print(f"Step {i}: ({wx:.3f}, {wy:.3f})  yaw={math.degrees(yaw):.1f}°")
    time.sleep(step_delay)

print("\nNavigation complete — robot is at goal")

Step 0: (1.000, 2.000)  yaw=0.0°
Step 1: (1.500, 2.000)  yaw=0.0°

Navigation complete — robot is at goal
